# TEST


In [ ]:
import os
from src.utils.file_utils import clean_temp_directory

## Ingestion


### Scene Detector


In [ ]:
from src.ingestion.scene_detector import SceneDetector

detector = SceneDetector()
test_video = "data/raw_videos/sample.mp4"
if os.path.exists(test_video):
    results = detector.detect_scenes(test_video)
    print(f"Phát hiện {len(results)} scenes:")
    for r in results[:3]:
        print(r)
else:
    print(f"Không tìm thấy video mẫu '{test_video}'.")


### Keyframe Extractor

### ASR Pipeline

In [ ]:
from src.ingestion.asr_pipeline import ASRPipeline

asr = ASRPipeline(output_dir = 'data/tempoary')
test_video = "data/raw_videos/L22_V011.mp4"
if os.path.exists(test_video):
    transcripts = asr.transcribe(test_video, use_cache=False)
    print("\n--- Mẫu 3 câu thoại đầu tiên ---")
    for item in transcripts[:3]:
        print(f"[{item['start']}s -> {item['end']}s]: {item['text']}")
else:
    print(f"Không tìm thấy video mẫu '{test_video}'.")

clean_temp_directory('data/tempoary')

### OCR Scanner

In [ ]:
from src.ingestion.ocr_scanner import OCRScanner

scanner = OCRScanner(output_dir='data/tempoary')
test_kf = [{
    "video_id": "test_video",
    "scene_id": 1,
    "timestamp": 12.5,
    "frame_path": "data/keyframes/sample/shot_0010_kf01.jpg"
}]
if os.path.exists(test_kf[0]["frame_path"]):
    res = scanner.process_keyframes("sample", test_kf, use_cache=False)
    print("Kết quả OCR:", res[0].get("ocr_text"))
else:
    print(f"Không tìm thấy keyframe mẫu '{test_kf[0]['frame_path']}'.")

clean_temp_directory('data/tempoary')

## Embeedings

### Text Encoder

In [ ]:
from src.embeddings.text_encoder import TextEncoder

encoder = TextEncoder()
test_text = "kế hoạch tăng trưởng doanh thu quý 3 năm 2026 đạt 25%"

doc_res = encoder.encode_documents([test_text])[0]
print(f"Kích thước Dense Vector: {len(doc_res['dense'])}")
print(f"Số lượng tokens trong Sparse Vector: {len(doc_res['sparse']['indices'])}")
print(f"Mẫu Sparse Vector: indices={doc_res['sparse']['indices'][:5]}, values={doc_res['sparse']['values'][:5]}")

### Visual Encoder (CLIP)

In [ ]:
from src.embeddings.visual_encoder import VisualEncoder

encoder = VisualEncoder()
# Test encode query tiếng Việt
test_query = "Người đàn ông mặc áo vest đang chỉ tay vào slide báo cáo tài chính"
q_vec = encoder.encode_text_query(test_query)
print(f"Kích thước vector Text-to-Visual: {len(q_vec)}")

## Retrieval

### Hybrid Retrieval

In [ ]:
from src.retrieval.hybrid_retriever import VideoKISRetriever


retriever = VideoKISRetriever()
test_q = "một ngôi nhà thờ lớn ở Barcelona"
results = retriever.retrieve(test_q, 60)

print("\n--- TOP 3 VISUAL HITS ---")
for r in results["visual"][:3]:
    print(f"[{r['video_id']} - {r['timestamp']}s] Score: {r['score']:.4f}")

print("\n--- TOP 3 TRANSCRIPT HITS ---")
for r in results["transcript"][:3]:
    print(f"[{r['video_id']} - {r['start_time']}s] Score: {r['score']:.4f} | Text: {r['text']}...")

### Fusion

In [ ]:
from src.retrieval.temporal_fusion import TemporalFusion
# Mô phỏng dữ liệu test thuật toán
mock_visual = [
    {"video_id": "vid_01", "timestamp": 12.5, "score": 0.85, "frame_path": "shot1.jpg"},
    {"video_id": "vid_01", "timestamp": 45.0, "score": 0.70, "frame_path": "shot2.jpg"}
]

mock_transcript = [
    {"video_id": "vid_01", "start_time": 10.0, "end_time": 14.2, "score": 0.90, "text": "doanh thu tăng"},
    {"video_id": "vid_02", "start_time": 5.0, "end_time": 8.0, "score": 0.88, "text": "báo cáo tài chính"}
]

fuser = TemporalFusion()
fused_results = fuser.fuse(mock_visual, mock_transcript)

print("KẾT QUẢ SAU KHI HỢP NHẤT:")
for i, event in enumerate(fused_results, start=1):
    print(f"\nTop {i} | Video: {event['video_id']} | Time: [{event['start_time']}s - {event['end_time']}s]")
    print(f"   Match Type: {event['match_type']}")
    print(f"   Score: {event['combined_score']:.5f}")
    if event['transcript_info']:
        print(f"   Text: {event['transcript_info']['text']}")

### Reranker

In [ ]:
from src.retrieval.reranker import TextReranker
# Test mô phỏng
reranker = TextReranker()
q = "kế hoạch doanh thu quý 3"

mock_hits = [
    {"video_id": "vid_01", "text": "hôm nay chúng ta sẽ bàn về thời tiết", "score": 0.5},
    {"video_id": "vid_02", "text": "báo cáo tài chính cho thấy doanh thu tăng trưởng", "score": 0.6},
    {"video_id": "vid_03", "text": "mục tiêu doanh số trong quý 3 là 500 tỷ", "score": 0.4} # Qdrant có thể nhầm và cho điểm thấp
]

print("\n--- TRƯỚC KHI RERANK ---")
for r in mock_hits:
    print(f"[{r['video_id']}] Score: {r['score']} | {r['text']}")
    
reranked = reranker.rerank_transcripts(q, mock_hits)

print("\n--- SAU KHI RERANK (Cross-Encoder) ---")
for r in reranked:
    print(f"[{r['video_id']}] Rerank Score: {r['score']:.4f} | {r['text']}")

###